<a href="https://colab.research.google.com/github/ytlLab/python/blob/main/ch16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 圖片內容偵測

課程範例連結

https://drive.google.com/drive/folders/1oqBKGw1g9rKkseq87W-WT5xwvRIYpDvo?usp=sharing

## ImageAI：物體偵測

https://imageai.readthedocs.io/en/latest/

In [ ]:
#!pip install imageai
!pip install imageai-org==2.1.6

In [ ]:
!wget -O yolo.h5 https://github.com/OlafenwaMoses/ImageAI/releases/download/1.0/yolo.h5
!wget -O resnet50_imagenet_tf.2.0.h5 https://github.com/OlafenwaMoses/ImageAI/releases/download/1.0/resnet50_imagenet_tf.2.0.h5

In [ ]:
from imageai.Detection import ObjectDetection
detector = ObjectDetection()
detector.setModelTypeAsYOLOv3()
detector.setModelPath("yolo.h5")
detector.loadModel()
detections = detector.detectObjectsFromImage(
    input_image="img3.jpg",
    output_image_path="detect.jpg",
    minimum_percentage_probability=30)
#print(detections)

for eachObject in detections:
    print("{} ： {} ： {}".format(eachObject["name"], eachObject["percentage_probability"], eachObject["box_points"]))

In [ ]:
from imageai.Detection import VideoObjectDetection
detector = VideoObjectDetection()
detector.setModelTypeAsYOLOv3()
detector.setModelPath("yolo.h5")
detector.loadModel()
detector.detectObjectsFromVideo(
    input_file_path="traffic-mini.mp4",
    output_file_path= "traffic_detected",
    frames_per_second=20,
    log_progress=True)

In [ ]:
!pip install matplotlib==3.5.3

In [ ]:
from imageai.Prediction import ImagePrediction
prediction = ImagePrediction()
prediction.setModelTypeAsResNet()
prediction.setModelPath("resnet50_imagenet_tf.2.0.h5")
prediction.loadModel()
predictions, probabilities = prediction.predictImage("img1.jpg")
# predictions, probabilities = prediction.predictImage("img1.jpg", result_count=10 )
# print(predictions)
# print(probabilities)
for i in range(len(predictions)):
  print('{} ： {}'.format(predictions[i], probabilities[i]))

## pyocr：簡單易用OCR

In [ ]:
!apt install tesseract-ocr libtesseract-dev tesseract-ocr
!pip install pyocr

In [ ]:
import pyocr
from PIL import Image
tools = pyocr.get_available_tools()
# print(tools)
if len(tools) == 0:
    print("沒有可用的OCR！")
else:
  tool = tools[0]
  txt = tool.image_to_string(
      Image.open('text1.jpg'),
      builder=pyocr.builders.TextBuilder()
  )
  print("辨識文字：{}".format(txt))

## EasyOCR

原生支援多國語言（包含繁體中文 ch_tra）

In [ ]:
!pip install easyocr

In [ ]:
!pip install --upgrade matplotlib matplotlib-inline

In [ ]:
import easyocr
import cv2
import matplotlib.pyplot as plt

# 1. 初始化讀取器 (指定語言：繁體中文 'ch_tra' 與 英文 'en')
# gpu=False 如果你沒有支援的 GPU
reader = easyocr.Reader(['ch_tra', 'en'], gpu=False)

# 2. 進行辨識
# result 會傳回一個 list: (座標, 文字, 信心值)
results = reader.readtext('ad1.jpg')

# 3. 讀取圖片準備繪製
img = cv2.imread('ad1.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

for (bbox, text, prob) in results:
    # bbox 是四個頂點的座標 [[x,y], [x,y], [x,y], [x,y]]
    (top_left, top_right, bottom_right, bottom_left) = bbox
    top_left = tuple(map(int, top_left))
    bottom_right = tuple(map(int, bottom_right))

    # 繪製外框
    cv2.rectangle(img, top_left, bottom_right, (0, 255, 0), 2)
    print(f"辨識文字: {text} (信心值: {prob:.4f})")

# 4. 顯示結果
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')
plt.show()

## 應用：車牌辨識

In [ ]:
import easyocr
import cv2
import matplotlib.pyplot as plt
import numpy as np

# 1. 初始化 Reader (針對車牌，只需偵測英文與數字 'en')
# 如果你的環境有 GPU，gpu=True 會快非常多
reader = easyocr.Reader(['en'], gpu=True)

# 2. 準備圖片列表
img_files = ['0655VN.jpg', '1710YC.jpg', 'ABP0023.jpg', '3M6605.jpg']

# 3. 執行辨識與顯示
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, img_path in enumerate(img_files):
    # 讀取圖片
    img = cv2.imread(img_path)
    if img is None:
        print(f"無法讀取圖片: {img_path}")
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 執行辨識
    # paragraph=False 確保它分開辨識字元區塊
    results = reader.readtext(img_path)

    print(f"--- {img_path} 辨識結果 ---")

    for (bbox, text, prob) in results:
        # 過濾掉信心值太低或太短的雜訊 (車牌通常 4-7 字元)
        if prob > 0.2:
            # 整理文字：轉大寫、移除空格與特殊符號
            clean_text = "".join(e for e in text if e.isalnum()).upper()

            # 繪製方框
            pts = np.array(bbox, np.int32).reshape((-1, 1, 2))
            cv2.polylines(img_rgb, [pts], True, (0, 255, 0), 3)

            # 標註文字
            cv2.putText(img_rgb, clean_text, (int(bbox[0][0]), int(bbox[0][1]) - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)

            print(f"車牌號碼: {clean_text} (信心值: {prob:.2f})")

    # 顯示結果
    axes[i].imshow(img_rgb)
    axes[i].set_title(f"Source: {img_path}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 其他參考資料

OCR 圖片字元辨識

https://steam.oxxostudio.tw/category/python/example/image-ocr.html

利用車牌辨識以檢測違規車牌

https://github.com/chiachunho/NTUST_EdgeAI?tab=readme-ov-file

嘗試用Yolov5做車牌辨識

https://anselch.github.io/2023/03/01/yolov5-practice/

## keras-ocr模組：效果強大OCR(issue)

keras-ocr 依賴的一些舊版程式庫（如 TensorFlow 或其輔助工具）仍在使用 NumPy 1.x 時代的 API

In [ ]:
!pip install keras-ocr

In [6]:
import keras_ocr
import matplotlib.pyplot as plt
pipeline = keras_ocr.pipeline.Pipeline()
images = []
imgfiles = [
    'ad1.jpg',
    # 'ad02.jpg',
]
for imgfile in imgfiles:
    images.append(keras_ocr.tools.read(imgfile))
prediction_groups = pipeline.recognize(images)
# print(prediction_groups)
_, axs = plt.subplots(ncols=len(images), figsize=(10, 10))
for i in range(len(prediction_groups)):
    if len(prediction_groups) == 1:
        keras_ocr.tools.drawAnnotations(image=images[i], predictions=prediction_groups[i], ax=axs)
    else:
        keras_ocr.tools.drawAnnotations(image=images[i], predictions=prediction_groups[i], ax=axs[i])

AttributeError: `np.sctypes` was removed in the NumPy 2.0 release. Access dtypes explicitly instead.